# test des fichiers fournis par la DMR

In [1]:
from shapely import LineString, Point, boundary
import geopandas as gpd
import geo_nx as gnx 
import networkx as nx 
import pandas as pd
import geo_nx.utils as utils
from networkx.utils import graphs_equal
import folium


## Génération des aires DMR

### Données initiales du 02/07/2026

In [32]:
strdate = '2026-07-08' # isoformat
data = 'data_DMR_02_07'

aires_c = gpd.read_file(f"../saturation/{data}/Polygones_Aires_C_Corrige.shp")
aires_nc = gpd.read_file(f"../saturation/{data}/Polygones_Aires_NC_Corrige.shp")
aires_nc["Réseau"] = "NC"
aires_nc["GVO_Axe"] = "NC"
aires = pd.concat([aires_c, aires_nc], ignore_index=True)
aires["id_aire"]="A" + (aires.index + 1).astype(str).str.zfill(6)
len(aires_c), len(aires_nc), len(aires)

(1072, 803, 1875)

In [ ]:
# aires.to_file(f"../saturation/{data}/aires_dmr_{strdate}.geojson", driver="GeoJSON")
# aires[['Aire', 'NomUsuel', 'Nature', 'GVO', 'Surface', 'Réseau', 'Axe']].to_csv("../saturation/data_DMR/aires_dmr.csv", index=False)

In [ ]:
aires_c_init=aires[aires["Aire"]=="RRN C"].copy()
aires_c_init_simple = aires_c_init[["NomUsuel", "Aire", "Nature", "GVO", "Réseau", "Axe", "GVO_Axe", "geometry"]]
#aires_c_init

In [6]:
#refnat = {'location': [46.3, 2.3], 'zoom_start': 7}
#aires.explore(m=folium.Map(**refnat), color='green', name='aires', show=True)

### Ajout des données du 03/08/2026

In [35]:
strdate = '2026-11-19' # isoformat
data = 'data_DMR_03_08'

aires_c_03_08 = gpd.read_file(f"../saturation/{data}/Polygones_Aires_C_Corrige.shp")
len(aires_c_03_08)

1069

In [8]:
aires_c_03_08[aires_c_03_08.duplicated(subset=["NomUsuel"], keep=False)].sort_values(by=["NomUsuel"])

,Aire,NomUsuel,Nature,GVO,Surface,Réseau,Axe,GVO_Axe,geometry
620,RRN C,Lusse,R,APRR,4591.75375,C,N0159,APRR,"POLYGON ((7.10812 48.29028, 7.10805 48.29036, ..."
621,RRN C,Lusse,R,APRR,8005.27274,C,N0159,APRR,"POLYGON ((7.10823 48.29018, 7.10879 48.2902, 7..."
687,RRN C,Mionnay,S,APRR,70036.29850,C,A0046,APRR,"POLYGON ((4.89358 45.88671, 4.89373 45.88672, ..."
688,RRN C,Mionnay,S,APRR,154946.32510,C,A0046,APRR,"POLYGON ((4.89597 45.884, 4.89524 45.88504, 4...."
967,RRN C,Myennes,P,APRR,5001.55359,C,A0077,APRR,"POLYGON ((2.9396 47.43598, 2.93955 47.43623, 2..."
968,RRN C,Myennes,P,APRR,8438.25032,C,A0077,APRR,"POLYGON ((2.94123 47.43782, 2.94245 47.43946, ..."
382,RRN C,Schwindratzheim Nord,P,SANEF,6043.15307,C,A0004,SANEF,"POLYGON ((7.59822 48.76947, 7.59821 48.76951, ..."
900,RRN C,Schwindratzheim Nord,P,SANEF,982.21810,C,A0004,SANEF,"POLYGON ((7.59845 48.7688, 7.59942 48.76884, 7..."


In [ ]:
# traitement des doublons de nom d'aires

# aire de Lusse 620 (la petite) à enlever
# ajouter Est à Mionay 688 (la grande) et Ouest à 687
# ajouter Est à Myennes 968 (la grande) et Ouest à 967
# renommer Schwindratzheim Nord 900 (la petite) en Schwindratzheim ext
aires_c_03_08.loc[967, "NomUsuel"] = "Myennes Ouest"
aires_c_03_08.loc[968, "NomUsuel"] = "Myennes Est"
aires_c_03_08.loc[688, "NomUsuel"] = "Mionay Est"
aires_c_03_08.loc[687, "NomUsuel"] = "Mionay Ouest"
aires_c_03_08.loc[900, "NomUsuel"] = "Schwindratzheim Ext"
aires_c_03_08.drop(index=620, inplace=True)

In [10]:
aires_c_03_08[aires_c_03_08["Aire"]=="RRNC"]

,Aire,NomUsuel,Nature,GVO,Surface,Réseau,Axe,GVO_Axe,geometry
1066,RRNC,Aire de Dijon-Spoy,S,APRR,136072.1567,C,A0031,APRR,"POLYGON ((5.17131 47.4244, 5.17194 47.4233, 5...."


In [11]:
# renommer les aires RRNC en RRN C
aires_c_03_08.loc[aires_c_03_08[aires_c_03_08["Aire"]=="RRNC"].index, "Aire"] = "RRN C"

In [12]:
aires_03_08 = aires.copy().set_index("id_aire", drop=True)
#aires_03_08

In [13]:
aires_03_08[aires_03_08.duplicated(subset=["NomUsuel"], keep=False)].sort_values(by=["NomUsuel"])

,Aire,NomUsuel,Nature,GVO,Surface,Réseau,Axe,GVO_Axe,geometry
id_aire,,,,,,,,,
A001733,RRN NC,Aire de Perles-et-Castelet (sens 1),R,DIRSO,4031.62069,NC,N0020,NC,"POLYGON ((1.78966 42.73835, 1.78975 42.73789, ..."
A001734,RRN NC,Aire de Perles-et-Castelet (sens 1),R,DIRSO,2282.27648,NC,N0020,NC,"POLYGON ((1.79024 42.73632, 1.79007 42.73674, ..."
A000605,RRN C,Chantemerle,R,COFIROUTE,38989.65540,C,A0028,COFIROUTE,"POLYGON ((0.61298 47.54215, 0.61554 47.53905, ..."
A001286,RRN NC,Chantemerle,S,DIRIF,9908.48043,NC,N0002,NC,"POLYGON ((2.71638 49.07102, 2.71655 49.07093, ..."
A000854,RRN C,Fleury,P,APRR,28421.09354,C,A0006,APRR,"POLYGON ((2.53979 48.42532, 2.54006 48.42464, ..."
A001117,RRN NC,Fleury,S,DIRIF,30683.31816,NC,N0104,NC,"POLYGON ((2.37173 48.62367, 2.37154 48.62359, ..."
A001118,RRN NC,La Prairie,S,DIRIF,7196.64675,NC,N0012,NC,"POLYGON ((1.63348 48.79203, 1.6346 48.79227, 1..."
A001248,RRN NC,La Prairie,S,DIRIF,3751.91655,NC,A0086,NC,"POLYGON ((2.23299 48.92508, 2.23409 48.92562, ..."
A000624,RRN C,Lusse,R,APRR,4591.75375,C,N0159,APRR,"POLYGON ((7.10812 48.29028, 7.10805 48.29036, ..."


In [14]:
# aire de Lusse A000624 (la petite) à enlever
# ajouter Est à Mionay A000693 (la grande) et Ouest à A000692
# ajouter Est à Myennes A000975 (la grande) et Ouest à A000974
# renommer Schwindratzheim Nord A000906 (la petite) en Schwindratzheim ext
aires_03_08.loc["A000974", "NomUsuel"] = "Myennes Ouest"
aires_03_08.loc["A000975", "NomUsuel"] = "Myennes Est"
aires_03_08.loc["A000692", "NomUsuel"] = "Mionay Ouest"
aires_03_08.loc["A000693", "NomUsuel"] = "Mionay Est"
aires_03_08.loc["A000906", "NomUsuel"] = "Schwindratzheim Ext"
aires_03_08.drop(index="A000624", inplace=True)

In [15]:
aires_c_03_08_init=aires_03_08[aires_03_08["Aire"]=="RRN C"].copy()
aires_c_03_08_init_simple = aires_c_03_08_init[["NomUsuel", "Aire", "Nature", "GVO", "Réseau", "Axe", "GVO_Axe", "geometry"]]

In [16]:
# analyse des différences entre les deux fichiers
diff = aires_c_03_08_init_simple.merge(
    aires_c_03_08,
    on="NomUsuel",
    how="outer",
    suffixes=("_init", "_new"),
    indicator=True
)
diff[diff["_merge"] != "both"][['NomUsuel', '_merge']]

,NomUsuel,_merge
201,Aire de Dijon-Brognon Nord,left_only
202,Aire de Dijon-Spoy,right_only
292,Aire des Terrasses de Provence Centre VL,left_only
484,Dijon-Spoy,left_only
615,LA VENDEE Est,left_only
630,LES PYRENEES SUD,right_only
632,LES TERRASSES DE PROVENCE CENTRE,left_only
633,LES TERRASSES DE PROVENCE,right_only
877,PYRENNEES SUD,left_only


Synthèse des modifs à effectuer: 

- dijon-spoy + dijon brognon nord -> aire dijon-spoy
  - maj du nom de dijon-spoy 484 en aire de dijon_spoy 202
  - suppression de aire de brognon nord 201
- terrasses centre VL + terrasse centre -> terrasses
  - maj du nom de aire des terrasses centre 292 en les terrasses de provence 633
  - suppression de les terrasses de provence centre 632
- suppression de vendée est 615
- maj du nom pyrennees sud 877 en les pyrennées sud 630

In [17]:
# maj dijon-spoy
id_aire = aires_03_08.loc[aires_03_08['NomUsuel'] == diff.loc[484]['NomUsuel']].index[0]
aires_03_08.loc[id_aire, 'NomUsuel'] = diff.loc[202, 'NomUsuel']
# del brognon nord
aires_03_08.drop(index=aires_03_08[aires_03_08['NomUsuel'] == diff.loc[201]['NomUsuel']].index[0], inplace=True)
# maj aire des terrasses centre
id_aire = aires_03_08.loc[aires_03_08['NomUsuel'] == diff.loc[292]['NomUsuel']].index[0]
aires_03_08.loc[id_aire, 'NomUsuel'] = diff.loc[633, 'NomUsuel']
# del les terrasses de provence centre
aires_03_08.drop(index=aires_03_08[aires_03_08['NomUsuel'] == diff.loc[632]['NomUsuel']].index[0], inplace=True)
# del les terrasses de vendee est
aires_03_08.drop(index=aires_03_08[aires_03_08['NomUsuel'] == diff.loc[615]['NomUsuel']].index[0], inplace=True)
# maj pyrennees sud
id_aire = aires_03_08.loc[aires_03_08['NomUsuel'] == diff.loc[877]['NomUsuel']].index[0]
aires_03_08.loc[id_aire, 'NomUsuel'] = diff.loc[630, 'NomUsuel']

In [18]:
# vérification des différences entre les deux fichiers
diff2 = aires_03_08[aires_03_08['Aire'] == "RRN C"].merge(
    aires_c_03_08,
    on="NomUsuel",
    how="outer",
    suffixes=("_init", "_new"),
    indicator=True
)
diff2[diff2["_merge"] != "both"][['NomUsuel', '_merge']]

,NomUsuel,_merge


In [19]:
aires_idem = diff2[diff2["_merge"] == "both"][['NomUsuel', 'Aire_init', 'Nature_init', 'GVO_init', 
       'Réseau_init', 'Axe_init', 'GVO_Axe_init', 'geometry_init']].copy().rename(columns={'Aire_init': 'Aire', 'Nature_init': 'Nature', 'GVO_init': 'GVO',
       'Réseau_init': 'Réseau', 'Axe_init': 'Axe', 'GVO_Axe_init': 'GVO_Axe', 'geometry_init': 'geometry'})
aires_idem_03_08 = diff2[diff2["_merge"] == "both"][['NomUsuel', 'Aire_new', 'Nature_new', 'GVO_new',
       'Réseau_new', 'Axe_new', 'GVO_Axe_new', 'geometry_new']].copy().rename(columns={'Aire_new': 'Aire', 'Nature_new': 'Nature', 'GVO_new': 'GVO',
       'Réseau_new': 'Réseau', 'Axe_new': 'Axe', 'GVO_Axe_new': 'GVO_Axe', 'geometry_new': 'geometry'})
ecart_idem = aires_idem.compare(aires_idem_03_08, align_axis=1)
ecart_idem

Nature                                                 geometry  \
       self other                                               self   
200     NaN   NaN  POLYGON ((5.17335 47.42079, 5.17296 47.42071, ...   
201       R     S  POLYGON ((5.1706 47.42635, 5.17272 47.42707, 5...   
211       S     R                                               None   
322     NaN   NaN  POLYGON ((1.53066 44.2306, 1.53259 44.22946, 1...   
323     NaN   NaN  POLYGON ((1.52887 44.23369, 1.53048 44.23203, ...   
625     NaN   NaN  POLYGON ((-0.08247 43.23324, -0.07999 43.23173...   
626     NaN   NaN  POLYGON ((-0.08143 43.23281, -0.07825 43.23104...   
628     NaN   NaN  POLYGON ((5.98965 43.42315, 5.99444 43.42282, ...   
780     NaN   NaN  POLYGON ((6.65941 49.13354, 6.65543 49.13417, ...   
781     NaN   NaN  POLYGON ((6.65816 49.13324, 6.65825 49.13328, ...   
857     NaN   NaN  POLYGON ((1.52501 45.03599, 1.52568 45.03464, ...   
858     NaN   NaN  POLYGON ((1.52565 45.0346, 1.52614 45.03368, 1...   
1007    NaN   NaN  POLYGON ((2.84549 42.57491, 2.84451 42.57438, ...   
1008    NaN   NaN  POLYGON ((2.84497 42.58032, 2.84476 42.57969, ...   

                                                         
                                                  other  
200   POLYGON ((5.17331 47.42073, 5.1732 47.4207, 5....  
201   POLYGON ((5.17131 47.4244, 5.17194 47.4233, 5....  
211                                                None  
322   POLYGON ((1.53064 44.23046, 1.53099 44.23022, ...  
323   POLYGON ((1.52978 44.23404, 1.53138 44.23267, ...  
625   POLYGON ((-0.08266 43.23178, -0.08277 43.23212...  
626   POLYGON ((-0.07902 43.23152, -0.0781 43.23103,...  
628   POLYGON ((5.99027 43.42192, 5.99033 43.42206, ...  
780   POLYGON ((6.65941 49.13354, 6.65565 49.13412, ...  
781   POLYGON ((6.6578 49.13341, 6.65788 49.13345, 6...  
857   POLYGON ((1.52501 45.03599, 1.52584 45.034, 1....  
858   POLYGON ((1.52584 45.034, 1.52614 45.03368, 1....  
1007  POLYGON ((2.84549 42.57491, 2.84451 42.57438, ...  
1008  POLYGON ((2.84951 42.57895, 2.84944 42.5777, 2...

In [ ]:
# mise à jour des aires identiques entre les deux fichiers
for idx in ecart_idem.index:
    id_aire = aires_03_08[aires_03_08['NomUsuel'] == aires_idem.loc[idx]['NomUsuel']].index[0]
    aires_03_08.loc[id_aire, :] = aires_idem_03_08.loc[idx, :]
    #print(idx, f"Updated id_aire {id_aire} with {aires_idem_03_08.loc[idx, :][['Aire', 'geometry']]} : {aires_03_08.loc[id_aire, :][['Aire', 'geometry']]}")
    print(idx, f"Updated id_aire {id_aire} with {aires_idem_03_08.loc[idx, :]['NomUsuel']}")
    

200 Updated id_aire A001072 with Aire de Dijon-Brognon
201 Updated id_aire A000491 with Aire de Dijon-Spoy
211 Updated id_aire A000152 with Aire de Lunel
322 Updated id_aire A001034 with BOIS DE DOURRE Flux 1
323 Updated id_aire A000687 with BOIS DE DOURRE Flux 2
625 Updated id_aire A000728 with LES PYRENEES
626 Updated id_aire A001056 with LES PYRENEES SUD
628 Updated id_aire A000664 with LES TERRASSES DE  PROVENCE
780 Updated id_aire A000942 with Longeville-les-Saint-Avold Nord
781 Updated id_aire A000221 with Longeville-les-Saint-Avold Sud
857 Updated id_aire A000319 with PECH MONTAT Flux 1
858 Updated id_aire A001033 with PECH MONTAT Flux 2
1007 Updated id_aire A000093 with VILLAGE CATALAN EST
1008 Updated id_aire A001031 with VILLAGE CATALAN OUEST


In [ ]:
# vérification de l'absence de différences entre les deux fichiers
diff3 = aires_03_08[aires_03_08['Aire'] == "RRN C"].merge(
    aires_c_03_08,
    on="NomUsuel",
    how="outer",
    suffixes=("_init", "_new"),
    indicator=True
)
diff3[diff3["_merge"] != "both"][['NomUsuel', '_merge']]

,NomUsuel,_merge


In [ ]:
# vérification de l'absence de différences des attributs entre les deux fichiers
aires_idem3 = diff3[diff3["_merge"] == "both"][['NomUsuel', 'Aire_init', 'Nature_init', 'GVO_init', 
       'Réseau_init', 'Axe_init', 'GVO_Axe_init', 'geometry_init']].copy().rename(columns={'Aire_init': 'Aire', 'Nature_init': 'Nature', 'GVO_init': 'GVO',
       'Réseau_init': 'Réseau', 'Axe_init': 'Axe', 'GVO_Axe_init': 'GVO_Axe', 'geometry_init': 'geometry'})
aires_idem3_03_08 = diff3[diff3["_merge"] == "both"][['NomUsuel', 'Aire_new', 'Nature_new', 'GVO_new',
       'Réseau_new', 'Axe_new', 'GVO_Axe_new', 'geometry_new']].copy().rename(columns={'Aire_new': 'Aire', 'Nature_new': 'Nature', 'GVO_new': 'GVO',
       'Réseau_new': 'Réseau', 'Axe_new': 'Axe', 'GVO_Axe_new': 'GVO_Axe', 'geometry_new': 'geometry'})
ecart_idem3 = aires_idem3.compare(aires_idem3_03_08, align_axis=1)
ecart_idem3

Empty DataFrame
Columns: []
Index: []

In [43]:
aires_03_08 = aires_03_08.reset_index()
aires_03_08["coordonnees"] = aires_03_08.to_crs(epsg=2154).geometry.centroid.to_crs(epsg=4326)

In [ ]:
echangeurs_c = gpd.read_file("../saturation/data_DMR_02_07/Echangeurs_RRNc_Corrige.shp")
echangeurs_c["id_aire"]="A" + (echangeurs_c.index + 1).astype(str).str.zfill(6)
echangeurs_c

In [ ]:
carrefours_nc = gpd.read_file("../saturation/data_DMR_02_07/Carrefours_RRNnc_Corrige.shp")
carrefours_nc["id_aire"]="A" + (carrefours_nc.index + 1).astype(str).str.zfill(6)
carrefours_nc

In [ ]:
refnat = {'tiles': 'cartodbpositron', 'location': [46.3, 2.3], 'zoom_start': 7}
aires_c_03_08.explore(m=folium.Map(**refnat), color='green', name='aires', show=True)

In [ ]:
gr_aires = gnx.from_geopandas_nodelist(aires, node_attr=True)
nx.set_node_attributes(gr_aires, "aires", name="nature")

In [ ]:
echangeurs = gpd.read_file("../saturation/data_DMR/echangeurs_rondpoint/3b4a0c4e-d5aa-4fc2-b721-86796e791c5a_echangeurs_2025-01-01.shp")
echangeurs

In [ ]:
echangeurs.columns
echangeurs[['objetID', 'dateRefere', 'route', 'pr', 'depPr', 'concession', 'abs',
       'cumul', 'x', 'y', 'z', 'cote', 'voie', 'accrochage', 'dateMES', 'nom',
       'No', 'nbBretelle', 'nature', 'nature_lib', 'forme', 'forme_lib',
       'fonction', 'commentair', 'sectionnem', 'sectionn_1']].to_csv("../saturation/data_DMR/echangeurs_dmr.csv", index=False)

In [ ]:
gr_ech = gnx.from_geopandas_nodelist(echangeurs, node_attr=True)
nx.set_node_attributes(gr_ech, "echangeurs", name="nature")

In [ ]:
rtet_path = '../rtet/'
data_path = '../data/'
version = 'V1'
gr_rtet = gnx.from_geopandas_edgelist(gpd.read_file(rtet_path + version + '_rtet_edge.geojson'), 
                                 node_gdf=gpd.read_file(rtet_path + version + '_rtet_node.geojson'),
                                 node_attr=True, edge_attr=True)

In [ ]:
refnat = {'tiles': 'cartodbpositron', 'location': [46.3, 2.3], 'zoom_start': 7}
#param_exp_nd = {'n_marker_kwds': {'radius': 3, 'fill': True}}
param_exp_rtet = {'e_popup': ['weight', 'source', 'target'], 'e_tooltip': ["source", "target"], 
                'n_tooltip': ["node_id", "nature"], 'n_marker_kwds': {'radius': 1, 'fill': False}}
#param_exp_gr = {'e_popup': ['weight', 'source', 'target', 'NATURE'], 'e_tooltip': ["source", "target"], 
#                'n_tooltip': ["node_id", "nature"], 'n_marker_kwds': {'radius': 1, 'fill': False}}
param_exp_aires = {'n_tooltip': ["node_id", "nature"], 'n_marker_kwds': {'radius': 1, 'fill': False}}
param_exp_ech = {'n_name': 'echangeur', 'n_color': 'blue', 'n_marker_kwds': {'radius': 3, 'fill': True}}

# carte = gr_bret.explore(refmap=refnat, n_color='black', e_name='edges bretelles', n_name='nodes bretelles', **param_exp_gr)
carte = gr_ech.explore(refmap=refnat, edges=False, **param_exp_ech)
carte = gr_aires.explore(refmap=carte, e_color='green', edges=False, n_name='nodes aires', n_color='black', **param_exp_aires)
#carte = aires.explore(m=carte, color='green', name='aires', tooltip=['nature'], show=True)
carte = aires.explore(m=carte, color='green', name='aires', show=True)
#carte = gr_nat.explore(refmap=carte, e_color='magenta', e_name='edges nat', n_name='nodes nat',  n_color='black', **param_exp_gr)
#carte = afir.gr_echangeur.explore(refmap=carte, edges=False, **param_exp_ech)
carte = gr_rtet.explore(refmap=carte, e_color='cyan', e_name='edges rtet', n_name='nodes rtet', layercontrol=True, n_color='black', **param_exp_rtet)
carte

In [ ]:
e5 = pd.DataFrame({'target': ['00'], 'value': [3], 'code': ['e5'], 'level': [0], 'period': ['d'], 'timestamp': ['2024-12-31T00:00:00+00:00'], 'category': [None], 
                   'extras': [{'id_pdc_itinerance': ['pdc1', 'pdc2', 'pdc3'], 
                               'id_station_itinerance': ['st1', 'st1', 'st2'],
                               'puissance_nominale': [50, 100, 150],
                               'latitude': [43.4159, 48.8566, 43.6045],
                               'longitude': [6.45082, 2.3522, 1.4442]}]})
station_pdc = pd.DataFrame(e5['extras'][0])
stations = station_pdc.drop_duplicates(subset=['id_station_itinerance']).reset_index(drop=True)
stations_gpd = gpd.GeoDataFrame(
    stations, 
    geometry=gpd.points_from_xy(stations['longitude'], stations['latitude']),
    crs="EPSG:4326"  # Le système de coordonnées standard GPS (WGS84)
)[['id_station_itinerance', 'geometry']]
stations_gpd

In [ ]:
aires_with_stations = gpd.sjoin(aires, station_pdc, how="left", predicate='contains')[['id_aire', 'id_station_itinerance', 'id_pdc_itinerance', 'puissance_nominale']]
aires_with_stations

In [ ]:
from datetime import date, timedelta

start = date.today()
start.isoformat()
start + timedelta(days=1)